# 05 - Feature Engineering para Machine Learning

Ate aqui (notebooks 01 a 04) eu fiquei so na engenharia de dados: coletar, limpar, transformar e organizar tudo em Bronze, Silver e Gold. Agora a historia muda de fase.

Antes de eu conseguir treinar um modelo, preciso "alimentar" ele com informacao relevante -- e nao adianta jogar os dados crus la dentro, porque um modelo de ML nao entende contexto sozinho, ele so enxerga numeros nas colunas. Essa etapa de transformar dado bruto em coluna que realmente ajuda o modelo a aprender se chama **Feature Engineering**.

Duas palavras que vou usar bastante daqui pra frente:
- **Feature**: qualquer coluna que serve de pista pro modelo. Ex: a media de preco dos ultimos 5 dias pode ajudar a entender pra onde a acao esta indo.
- **Target**: a coluna que eu quero que o modelo aprenda a prever. Aqui, e o retorno do dia seguinte.

O plano deste notebook:
1. Ler os dados ja limpos da Silver
2. Criar features de medias moveis e volatilidade
3. Dar "memoria" ao modelo com os retornos passados (lag)
4. Criar a coluna target (retorno do dia seguinte)
5. Salvar tudo numa tabela Gold nova, pronta pro notebook de treino

In [0]:
from pyspark.sql import functions as F, Window

# Leio a tabela Silver, que ja esta limpa e com o retorno diario calculado
# (isso foi feito la no notebook 02_silver_transform)
df_silver = spark.table("b3_pipeline.silver_b3_stocks")

# printSchema() so mostra nome e tipo de cada coluna, sem processar nada pesado
df_silver.printSchema()

## Passo 1 -- Recriando a "janela" por ticker

Ja usei Window no notebook 02, mas vale reforcar porque aqui ele e a peca central de tudo: tenho 10 acoes diferentes na mesma tabela, uma linha por dia por acao. Se eu nao separar isso, o Spark vai calcular media movel misturando PETR4 com MGLU3, o que nao faz o menor sentido.

- `partitionBy("ticker")` -- trata cada acao isoladamente
- `orderBy("date")` -- dentro de cada acao, respeita a ordem cronologica, ja que media movel e retorno passado so existem se eu souber o que veio antes

In [0]:
# w = a janela base: uma acao por vez, sempre em ordem de data
w = Window.partitionBy("ticker").orderBy("date")

## Passo 2 -- Medias moveis

Media movel de 5 dias e basicamente: pego o preco de fechamento de hoje e dos 4 dias anteriores, e tiro a media. E um dos indicadores mais usados por quem opera na bolsa (eu mesmo usava isso antes de estudar dados) porque suaviza o ruido do dia a dia e mostra melhor a tendencia.

`rowsBetween(-4, 0)` define esse intervalo em relacao a linha atual: das 4 linhas anteriores ate a atual, ou seja, 5 linhas no total. O mesmo raciocinio vale pra `rowsBetween(-9, 0)`, que pega as ultimas 10.

In [0]:
# Janelas de tamanho fixo: 5 dias e 10 dias, sempre olhando pra tras (passado)
w_5d = w.rowsBetween(-4, 0)    # dia atual + 4 dias anteriores = 5 dias
w_10d = w.rowsBetween(-9, 0)   # dia atual + 9 dias anteriores = 10 dias

df_features = df_silver \
    .withColumn("media_movel_5d", F.avg("close").over(w_5d)) \
    .withColumn("media_movel_10d", F.avg("close").over(w_10d))

## Passo 3 -- Volatilidade recente

No notebook 03 eu calculei a volatilidade por setor, mas ali era so uma foto parada, um numero fixo por setor. Aqui eu quero outra coisa: a volatilidade recente de cada acao, dia a dia, pro modelo enxergar se ela andou instavel ou tranquila nos ultimos dias.

Uso `stddev` (desvio padrao) em cima da mesma janela de 5 dias que criei no passo anterior.

In [0]:
df_features = df_features \
    .withColumn("volatilidade_5d", F.stddev("daily_return_pct").over(w_5d))

## Passo 4 -- Dando "memoria" ao modelo (lag)

Um modelo de ML nao lembra de nada sozinho -- ele so enxerga o que esta escrito nas colunas daquela linha especifica. Se eu quiser que ele "saiba" o que aconteceu nos dias anteriores, preciso trazer esse valor explicitamente pra linha de hoje. Isso e o que chamam de lag (atraso).

`F.lag("daily_return_pct", 3)` pega o retorno de 3 linhas atras (sempre dentro da mesma janela `w`, ou seja, do mesmo ticker) e cola na linha atual.

In [0]:
df_features = df_features \
    .withColumn("retorno_lag1", F.lag("daily_return_pct", 1).over(w)) \
    .withColumn("retorno_lag3", F.lag("daily_return_pct", 3).over(w)) \
    .withColumn("retorno_lag5", F.lag("daily_return_pct", 5).over(w))

## Passo 5 -- Criando o target (o que eu quero prever)

Essa e a parte que realmente transforma isso num problema de Machine Learning. Eu quero prever o retorno de amanha, e pra treinar um modelo supervisionado cada linha do passado precisa vir junto com o resultado real que ja aconteceu -- e claro que eu ja sei o retorno de ontem, o modelo e que precisa aprender esse padrao.

`F.lead` e o inverso do lag: em vez de olhar pra tras, olha pra frente. `F.lead("daily_return_pct", 1)` traz o retorno do dia seguinte pra linha de hoje. Essa coluna, `target_retorno_prox_dia`, e o que o modelo vai tentar acertar usando as outras features.

In [0]:
df_features = df_features \
    .withColumn("target_retorno_prox_dia", F.lead("daily_return_pct", 1).over(w))

## Passo 6 -- Tirando as linhas incompletas

Duas situacoes deixam `null` nas colunas novas:

- No comeco de cada acao, ainda nao tem historico suficiente -- a primeira linha, por exemplo, nao tem `retorno_lag5` porque nao existem 5 dias anteriores pra olhar.
- No final de cada acao (a data mais recente), nao existe "dia seguinte" ainda, entao `target_retorno_prox_dia` fica vazio.

Modelo de ML nao roda com valor nulo no meio do caminho, entao seleciono as colunas finais e uso `.na.drop()` pra descartar qualquer linha incompleta.

In [0]:
df_features_final = df_features.select(
    "ticker", "setor", "date", "close", "volume",
    "daily_return_pct",
    "media_movel_5d", "media_movel_10d", "volatilidade_5d",
    "retorno_lag1", "retorno_lag3", "retorno_lag5",
    "target_retorno_prox_dia"
).na.drop()

## Passo 7 -- Salvando como tabela Gold

Pra fechar, salvo essa tabela no Delta Lake seguindo o mesmo padrao das outras tabelas Gold do projeto. Ela vai servir de "feature store": o proximo notebook, o de treino do modelo, vai ler direto daqui sem precisar recalcular nada de novo.

In [0]:
df_features_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("b3_pipeline.gold_ml_features")

print("Feature store salvo com sucesso!")
display(df_features_final)